In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine


BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DB_CONFIG = {
    "user": os.getenv("DB_USER", "root"),
    "password": os.getenv("DB_PASSWORD", "123456"),
    "host": os.getenv("DB_HOST", "localhost"),
    "port": os.getenv("DB_PORT", "3307"),
    "database": os.getenv("DB_NAME", "apple_repair_service"),
}

connection_url = (
    f"mysql+pymysql://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}?charset=utf8mb4"
)
engine = create_engine(connection_url)

SQL_QUERY = """
SELECT 
    st.id AS ticket_id,
    st.ticket_code,
    st.branch_id,
    cd.device_model_id,
    dm.name AS device_model_name,
    COALESCE(services_summary.total_services_count, 0) AS total_services_count,
    services_summary.service_issue_ids_list,
    services_summary.service_issue_names_list,
    services_summary.service_type_codes_list,
    COALESCE(services_summary.total_catalog_standard_duration_minutes, 0) AS total_catalog_standard_duration_minutes,
    COALESCE(parts_summary.total_parts_quantity_used, 0) AS total_parts_quantity_used,
    a_final.technician_id,
    u_tech.full_name AS technician_name,
    CASE WHEN COALESCE(assign_summary.total_assignments, 1) > 1 THEN 1 ELSE 0 END AS is_reassigned,
    COALESCE(pause_summary.count_waiting_part_events, 0) AS count_waiting_part_events,
    COALESCE(pause_summary.count_waiting_approval_events, 0) AS count_waiting_approval_events,
    COALESCE(pause_summary.total_pause_events_count, 0) AS total_pause_events_count,
    COALESCE(pause_summary.total_pause_duration_minutes, 0) AS total_pause_duration_minutes,
    st.actual_repair_started_at,
    st.actual_repair_completed_at,
    TIMESTAMPDIFF(MINUTE, st.actual_repair_started_at, st.actual_repair_completed_at) AS actual_repair_duration_minutes,
    (TIMESTAMPDIFF(MINUTE, st.actual_repair_started_at, st.actual_repair_completed_at) 
     - COALESCE(pause_summary.total_pause_duration_minutes, 0)) AS net_repair_duration_minutes
FROM service_tickets st
LEFT JOIN customer_devices cd ON st.customer_device_id = cd.id
LEFT JOIN device_models dm ON cd.device_model_id = dm.id
LEFT JOIN assignments a_final ON st.id = a_final.service_ticket_id AND a_final.status IN ('ENDED', 'ACTIVE')
LEFT JOIN users u_tech ON a_final.technician_id = u_tech.id
LEFT JOIN (
    SELECT service_ticket_id, COUNT(*) AS total_assignments
    FROM assignments GROUP BY service_ticket_id
) assign_summary ON st.id = assign_summary.service_ticket_id
LEFT JOIN (
    SELECT 
        ri.service_ticket_id,
        COUNT(DISTINCT ri.id) AS total_services_count,
        GROUP_CONCAT(DISTINCT s_issue.id ORDER BY s_issue.id SEPARATOR ',') AS service_issue_ids_list,
        GROUP_CONCAT(DISTINCT s_issue.name ORDER BY s_issue.id SEPARATOR ' + ') AS service_issue_names_list,
        GROUP_CONCAT(DISTINCT styp.code ORDER BY ri.id SEPARATOR ', ') AS service_type_codes_list,
        SUM(sci.standard_duration_minutes) AS total_catalog_standard_duration_minutes
    FROM repair_items ri
    LEFT JOIN service_catalog_items sci ON ri.service_catalog_item_id = sci.id
    LEFT JOIN service_types styp ON sci.service_type_id = styp.id
    LEFT JOIN service_issues s_issue ON sci.service_issue_id = s_issue.id
    GROUP BY ri.service_ticket_id
) services_summary ON st.id = services_summary.service_ticket_id
LEFT JOIN (
    SELECT 
        ri.service_ticket_id,
        SUM(COALESCE(pu.quantity, 0)) AS total_parts_quantity_used
    FROM repair_items ri
    INNER JOIN part_usages pu ON ri.id = pu.repair_item_id AND pu.status = 'ACTIVE'
    GROUP BY ri.service_ticket_id
) parts_summary ON st.id = parts_summary.service_ticket_id
LEFT JOIN (
    SELECT 
        service_ticket_id,
        SUM(CASE WHEN to_status = 'WAITING_PART' THEN 1 ELSE 0 END) AS count_waiting_part_events,
        SUM(CASE WHEN to_status = 'WAITING_APPROVAL' THEN 1 ELSE 0 END) AS count_waiting_approval_events,
        COUNT(*) AS total_pause_events_count,
        SUM(TIMESTAMPDIFF(MINUTE, changed_at, resolved_at)) AS total_pause_duration_minutes
    FROM service_status_histories
    WHERE to_status IN ('WAITING_PART', 'WAITING_APPROVAL') AND resolved_at IS NOT NULL
    GROUP BY service_ticket_id
) pause_summary ON st.id = pause_summary.service_ticket_id
WHERE st.actual_repair_started_at IS NOT NULL 
  AND st.actual_repair_completed_at IS NOT NULL
  AND dm.status = 'ACTIVE'
ORDER BY st.id DESC;
"""

df_raw = pd.read_sql(SQL_QUERY, con=engine)
print(f"Kich thuoc tap du lieu: {df_raw.shape}")
df_raw.head()

Kich thuoc tap du lieu: (858, 22)


,ticket_id,ticket_code,branch_id,device_model_id,device_model_name,total_services_count,service_issue_ids_list,service_issue_names_list,service_type_codes_list,total_catalog_standard_duration_minutes,total_parts_quantity_used,technician_id,technician_name,is_reassigned,count_waiting_part_events,count_waiting_approval_events,total_pause_events_count,total_pause_duration_minutes,actual_repair_started_at,actual_repair_completed_at,actual_repair_duration_minutes,net_repair_duration_minutes
0,11204,ST-202610-00847,1070,1445,iPad Pro M3,2,"1,6",Pin + Loa ngoài,"REPAIR, REPLACEMENT",105.0,0.0,5343,Nguyễn Minh Hoàng,0,0.0,1.0,1,20.0,2026-10-11 10:26:09,2026-10-11 11:56:09,90,70.0
1,11203,ST-202610-00846,1070,1444,iPad Pro M2,1,2,Camera sau,REPAIR,45.0,0.0,5344,Trần Quốc Huy,0,1.0,0.0,1,30.0,2026-10-11 10:26:09,2026-10-11 11:41:09,75,45.0
2,11202,ST-202610-00845,1070,1444,iPad Pro M2,1,3,Camera trước,REPAIR,45.0,0.0,5343,Nguyễn Minh Hoàng,0,1.0,1.0,2,54.0,2026-10-18 10:26:09,2026-10-18 12:07:09,101,47.0
3,11201,ST-202610-00844,1070,1444,iPad Pro M2,1,6,Loa ngoài,REPAIR,45.0,0.0,5343,Nguyễn Minh Hoàng,1,0.0,0.0,0,0.0,2026-10-18 10:26:09,2026-10-18 11:21:09,55,55.0
4,11201,ST-202610-00844,1070,1444,iPad Pro M2,1,6,Loa ngoài,REPAIR,45.0,0.0,5344,Trần Quốc Huy,1,0.0,0.0,0,0.0,2026-10-18 10:26:09,2026-10-18 11:21:09,55,55.0


In [35]:
COLS_TO_DROP = [
    "ticket_id",
    "ticket_code",
    "device_model_name",
    "technician_name",
    "service_issue_names_list",
    "service_type_codes_list",
    "actual_repair_started_at",
    "actual_repair_completed_at",
    "actual_repair_duration_minutes",
    "total_pause_duration_minutes",
    "total_pause_events_count",
    "count_waiting_part_events",
    "count_waiting_approval_events",
    "is_reassigned",
]

# Loai bo cac ca chuyen giao 
df_clean = df_raw[df_raw["is_reassigned"] == 0].copy() if "is_reassigned" in df_raw.columns else df_raw.copy()

# Xoa cac bien co the lam ro ri du lieu
drop_cols = [col for col in COLS_TO_DROP if col in df_clean.columns]
df_clean = df_clean.drop(columns=drop_cols)

print(f"Kich thuoc du lieu ban dau: {df_raw.shape}")
print(f"Kich thuoc du lieu sau khi clean: {df_clean.shape}")
print(f"Cac bien bi loai bo: {drop_cols}")

df_clean.head()

Kich thuoc du lieu ban dau: (858, 22)
Kich thuoc du lieu sau khi clean: (832, 8)
Cac bien bi loai bo: ['ticket_id', 'ticket_code', 'device_model_name', 'technician_name', 'service_issue_names_list', 'service_type_codes_list', 'actual_repair_started_at', 'actual_repair_completed_at', 'actual_repair_duration_minutes', 'total_pause_duration_minutes', 'total_pause_events_count', 'count_waiting_part_events', 'count_waiting_approval_events', 'is_reassigned']


,branch_id,device_model_id,total_services_count,service_issue_ids_list,total_catalog_standard_duration_minutes,total_parts_quantity_used,technician_id,net_repair_duration_minutes
0,1070,1445,2,"1,6",105.0,0.0,5343,70.0
1,1070,1444,1,2,45.0,0.0,5344,45.0
2,1070,1444,1,3,45.0,0.0,5343,47.0
5,1070,1444,2,"1,6",105.0,0.0,5344,70.0
8,1070,1444,2,"1,5",120.0,0.0,5344,85.0


In [ ]:
df_eng = df_clean.copy()

# 1. Thời gian chuẩn định mức trung bình cho 1 dịch vụ
df_eng["duration_per_service"] = (
    df_eng["total_catalog_standard_duration_minutes"] / df_eng["total_services_count"]
)

# 2. Số lượng linh kiện trung bình dùng cho 1 dịch vụ
df_eng["parts_per_service"] = (
    df_eng["total_parts_quantity_used"] / df_eng["total_services_count"]
)

print(f"✅ Đã tạo thành công 2 đặc trưng tỷ lệ nghiệp vụ mới.")
print(f"Kích thước DataFrame hiện tại: {df_eng.shape}")

df_eng[
    [
        "total_services_count",
        "total_catalog_standard_duration_minutes",
        "duration_per_service",
        "total_parts_quantity_used",
        "parts_per_service",
    ]
].head()

✅ Đã tạo thành công 2 đặc trưng tỷ lệ nghiệp vụ mới.
Kích thước DataFrame hiện tại: (832, 10)


,total_services_count,total_catalog_standard_duration_minutes,duration_per_service,total_parts_quantity_used,parts_per_service
0,2,105.0,52.5,0.0,0.0
1,1,45.0,45.0,0.0,0.0
2,1,45.0,45.0,0.0,0.0
5,2,105.0,52.5,0.0,0.0
8,2,120.0,60.0,0.0,0.0


In [32]:
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder

ISSUE_COL = "service_issue_ids_list"
CAT_COLS = ["device_model_id", "technician_id", "branch_id"]

# 1. MultiLabelBinarizer cho danh sách mã lỗi
parsed_issues = df_eng[ISSUE_COL].astype(str).apply(
    lambda x: [int(i.strip()) for i in x.split(",") if i.strip().isdigit()]
)

mlb = MultiLabelBinarizer()
issue_matrix = mlb.fit_transform(parsed_issues)
issue_cols = [f"issue_id_{cls}" for cls in mlb.classes_]
df_issues = pd.DataFrame(issue_matrix, columns=issue_cols, index=df_eng.index)

# 2. OneHotEncoder cho ID thiết bị, kỹ thuật viên và chi nhánh
ohe = OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore")
cat_matrix = ohe.fit_transform(df_eng[CAT_COLS])
ohe_cols = list(ohe.get_feature_names_out(CAT_COLS))
df_cat = pd.DataFrame(cat_matrix, columns=ohe_cols, index=df_eng.index)

# 3. Gộp bảng tính toán
df_encoded = pd.concat(
    [df_eng.drop(columns=[ISSUE_COL] + CAT_COLS), df_issues, df_cat], axis=1
)

print(f"Đã mã hóa dữ liệu thành công:")
print(f"   + Số đặc trưng mã lỗi mới (MultiLabel) : {len(issue_cols)} cột")
print(f"   + Số đặc trưng danh mục mới (One-Hot)  : {len(ohe_cols)} cột")
print(f"Kích thước DataFrame sau mã hóa       : {df_encoded.shape}")

df_encoded.head()

Đã mã hóa dữ liệu thành công:
   + Số đặc trưng mã lỗi mới (MultiLabel) : 7 cột
   + Số đặc trưng danh mục mới (One-Hot)  : 22 cột
Kích thước DataFrame sau mã hóa       : (832, 35)


,total_services_count,total_catalog_standard_duration_minutes,total_parts_quantity_used,net_repair_duration_minutes,duration_per_service,parts_per_service,issue_id_1,issue_id_2,issue_id_3,issue_id_4,issue_id_5,issue_id_6,issue_id_7,device_model_id_319,device_model_id_320,device_model_id_405,device_model_id_782,device_model_id_789,device_model_id_790,device_model_id_792,device_model_id_794,device_model_id_1415,device_model_id_1416,device_model_id_1417,device_model_id_1427,device_model_id_1428,device_model_id_1429,device_model_id_1440,device_model_id_1441,device_model_id_1442,device_model_id_1443,device_model_id_1444,device_model_id_1445,technician_id_5344,technician_id_5345
0,2,105.0,0.0,70.0,52.5,0.0,1,0,0,0,0,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,1,45.0,0.0,45.0,45.0,0.0,0,1,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
2,1,45.0,0.0,47.0,45.0,0.0,0,0,1,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
5,2,105.0,0.0,70.0,52.5,0.0,1,0,0,0,0,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
8,2,120.0,0.0,85.0,60.0,0.0,1,0,0,0,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0


In [34]:
# 1. Định vị đường dẫn lưu file trong thư mục data/processed/
OUTPUT_CSV_PATH = BASE_DIR / "data" / "processed" / "processed_ml_dataset_real.csv"
OUTPUT_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)

# 2. Ghi bảng dữ liệu đã hoàn thiện mã hóa ra tệp CSV
df_encoded.to_csv(OUTPUT_CSV_PATH, index=False)

print(f"🎉 [THÀNH CÔNG] Đã tạo dataset hoàn chỉnh từ CSDL thực tế!")
print(f"📁 Đường dẫn lưu tệp : {OUTPUT_CSV_PATH}")
print(f"📊 Kích thước đầu ra  : {df_encoded.shape} (Dòng x Cột)")

# Hiển thị 5 dòng đầu tiên làm minh họa báo cáo
df_encoded.head()

🎉 [THÀNH CÔNG] Đã tạo dataset hoàn chỉnh từ CSDL thực tế!
📁 Đường dẫn lưu tệp : d:\Class\DOAN\CNPM\apple-repair-service\apple-repair-service\ml\data\processed\processed_ml_dataset_real.csv
📊 Kích thước đầu ra  : (832, 35) (Dòng x Cột)


,total_services_count,total_catalog_standard_duration_minutes,total_parts_quantity_used,net_repair_duration_minutes,duration_per_service,parts_per_service,issue_id_1,issue_id_2,issue_id_3,issue_id_4,issue_id_5,issue_id_6,issue_id_7,device_model_id_319,device_model_id_320,device_model_id_405,device_model_id_782,device_model_id_789,device_model_id_790,device_model_id_792,device_model_id_794,device_model_id_1415,device_model_id_1416,device_model_id_1417,device_model_id_1427,device_model_id_1428,device_model_id_1429,device_model_id_1440,device_model_id_1441,device_model_id_1442,device_model_id_1443,device_model_id_1444,device_model_id_1445,technician_id_5344,technician_id_5345
0,2,105.0,0.0,70.0,52.5,0.0,1,0,0,0,0,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,1,45.0,0.0,45.0,45.0,0.0,0,1,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
2,1,45.0,0.0,47.0,45.0,0.0,0,0,1,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
5,2,105.0,0.0,70.0,52.5,0.0,1,0,0,0,0,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
8,2,120.0,0.0,85.0,60.0,0.0,1,0,0,0,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
